# Практична робота №15
## Критерії узгодженості й таблиці спряженості: χ²

**Виконав:** Панасюк Дмитро  
**Номер за списком:** 15  
**Варіант:** №5 — **Дніпро**  
**Базова середньорічна температура:** 10,0 °C  
**Сезонна амплітуда:** 14 °C

**Мета:** створити синтетичний кліматичний набір за 2021–2024 роки, побудувати таблицю спряженості, перевірити незалежність двох категоріальних ознак критерієм χ², перевірити його умови застосовності та дослідити критерії узгодженості із заявленими частками.

Для перевірки гіпотез приймаємо **рівень значущості α = 0,05**. Дані навчальні, а не реальні кліматичні спостереження.

### Підготовка: побудова набору `climate`

Згідно з умовою, використовуємо `np.random.seed(42)`, 4 роки й 12 місяців. Категорія відхилення визначається відносно **10 °C**: нижче −3 °C від бази — «холодніше», вище +3 °C — «тепліше», решта — «звичайно».

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare
from IPython.display import display

np.random.seed(42)
base_temp = 10.0
amplitude = 14
city = "Дніпро"
alpha = 0.05

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp

        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)
assert climate.shape == (48, 6)
assert climate["місто"].eq(city).all()
print("Форма набору:", climate.shape)
print("Перші 12 рядків:")
display(climate.head(12))

Форма набору: (48, 6)
Перші 12 рядків:


,місто,рік,місяць,температура,сезон,відхилення_від_норми
0,Дніпро,2021,1,-3.5,зима,холодніше
1,Дніпро,2021,2,-2.3,зима,холодніше
2,Дніпро,2021,3,3.6,весна,холодніше
3,Дніпро,2021,4,11.5,весна,звичайно
4,Дніпро,2021,5,16.8,весна,тепліше
5,Дніпро,2021,6,21.9,літо,тепліше
6,Дніпро,2021,7,25.6,літо,тепліше
7,Дніпро,2021,8,22.9,літо,тепліше
8,Дніпро,2021,9,16.5,осінь,тепліше
9,Дніпро,2021,10,10.5,осінь,звичайно


## Завдання 1. Таблиця спряженості власного варіанта

Будуємо абсолютні частоти та нормовану за рядками таблицю (`normalize="index"`).

In [2]:
season_order = ["зима", "весна", "літо", "осінь"]
cat_order = ["холодніше", "звичайно", "тепліше"]

observed = pd.crosstab(
    climate["сезон"], climate["відхилення_від_норми"]
).reindex(index=season_order, columns=cat_order, fill_value=0)

relative_by_season = pd.crosstab(
    climate["сезон"], climate["відхилення_від_норми"], normalize="index"
).reindex(index=season_order, columns=cat_order, fill_value=0)

print("Абсолютні частоти:")
display(observed)
print("Частки в межах сезону:")
display(relative_by_season.round(3))
print("Суми часток у кожному рядку:")
print(relative_by_season.sum(axis=1))

maximum = observed.to_numpy().max()
most_common = [
    f"{s} / {cat} ({observed.loc[s, cat]})"
    for s in observed.index for cat in observed.columns
    if observed.loc[s, cat] == maximum
]
print("Найчастіші комбінації:", "; ".join(most_common))

Абсолютні частоти:


відхилення_від_норми,холодніше,звичайно,тепліше
сезон,,,
зима,12,0,0
весна,4,4,4
літо,0,0,12
осінь,4,4,4


Частки в межах сезону:


відхилення_від_норми,холодніше,звичайно,тепліше
сезон,,,
зима,1.000,0.000,0.000
весна,0.333,0.333,0.333
літо,0.000,0.000,1.000
осінь,0.333,0.333,0.333


Суми часток у кожному рядку:
сезон
зима     1.0
весна    1.0
літо     1.0
осінь    1.0
dtype: float64
Найчастіші комбінації: зима / холодніше (12); літо / тепліше (12)


**Письмова відповідь.** Найбільшу частоту (по **12 спостережень**) мають одразу дві комбінації: **«зима — холодніше»** та **«літо — тепліше»**. Це очікувано: сезонна косинусоїда дає найбільші від’ємні відхилення взимку та додатні влітку.

У нормованій таблиці стоять **частки**, а не кількості. Наприклад, 1,000 у рядку «літо» та стовпці «тепліше» означає 100% літніх спостережень цього штучного набору. Сума рядка дорівнює 1, бо кожне спостереження належить рівно одній температурній категорії. Частки зручні для порівняння сезонів, особливо якби у них була різна кількість спостережень.

## Завдання 2. Критерій χ² незалежності

**H₀:** сезон і категорія відхилення від норми незалежні.  
**H₁:** між сезоном і категорією відхилення від норми є статистичний зв’язок.

Порівнюємо фактичні частоти з очікуваними за незалежності. Висновок за цим початковим тестом є **попереднім**, доки в Завданні 3 не перевірені очікувані частоти.

In [3]:
chi2_initial, p_initial, dof_initial, expected_array = chi2_contingency(observed)
expected = pd.DataFrame(
    expected_array, index=observed.index, columns=observed.columns
)
difference = observed - expected
absolute_difference = difference.abs()
max_abs_diff = absolute_difference.to_numpy().max()
biggest_cells = [
    f"{s} / {cat}: O={observed.loc[s, cat]}, E={expected.loc[s, cat]:.1f}"
    for s in observed.index for cat in observed.columns
    if np.isclose(absolute_difference.loc[s, cat], max_abs_diff)
]

print(f"χ² = {chi2_initial:.4f}, df = {dof_initial}, p-value = {p_initial:.9f}")
print("Очікувані частоти за H₀:")
display(expected.round(2))
print("Різниця observed − expected:")
display(difference.round(2))
print("Найбільше абсолютне відхилення:", max_abs_diff)
print("Клітинки:", "; ".join(biggest_cells))
print("Попереднє рішення:", "відхилити H₀" if p_initial < alpha else "не відхиляти H₀")

χ² = 38.4000, df = 6, p-value = 0.000000938
Очікувані частоти за H₀:


відхилення_від_норми,холодніше,звичайно,тепліше
сезон,,,
зима,5.0,2.0,5.0
весна,5.0,2.0,5.0
літо,5.0,2.0,5.0
осінь,5.0,2.0,5.0


Різниця observed − expected:


відхилення_від_норми,холодніше,звичайно,тепліше
сезон,,,
зима,7.0,-2.0,-5.0
весна,-1.0,2.0,-1.0
літо,-5.0,-2.0,7.0
осінь,-1.0,2.0,-1.0


Найбільше абсолютне відхилення: 7.0
Клітинки: зима / холодніше: O=12, E=5.0; літо / тепліше: O=12, E=5.0
Попереднє рішення: відхилити H₀


**Письмова відповідь.** Початковий χ² = **38,40**, p ≈ **9,38 · 10⁻⁷**, що менше за 0,05. Це вказує на сильний зв’язок у сконструйованому наборі: температурні категорії не розподілені рівномірно між сезонами.

Найбільша абсолютна різниця між спостереженою та очікуваною частотами — **7**: у клітинках **«зима — холодніше»** і **«літо — тепліше»** спостережено по 12 випадків замість 5 очікуваних.

**Застереження:** початкова таблиця має замалі очікувані частоти; тому наведене p-value не слід трактувати як надійний остаточний результат без перевірки та виправлення в наступному завданні.

## Завдання 3. Перевірка умови застосовності

Для стандартного χ²-наближення за правилом із завдання перевіряємо, чи всі очікувані клітинки мають **E ≥ 5**. Якщо ні, об’єднуємо змістовно суміжні за визначенням категорії: **«холодніше» + «звичайно» → «не тепліше»**. Після цього повторно тестуємо незалежність.

In [4]:
too_small = expected < 5
print("Матриця клітинок з очікуваною частотою < 5:")
display(too_small)

low_expected_cells = [
    (s, cat, float(expected.loc[s, cat]))
    for s in expected.index for cat in expected.columns
    if too_small.loc[s, cat]
]
print("Кількість клітинок з E < 5:", len(low_expected_cells))
for s, cat, e in low_expected_cells:
    print(f"  {s} / {cat}: E = {e:.1f}")

climate["відхилення_2_категорії"] = climate["відхилення_від_норми"].replace({
    "холодніше": "не тепліше", "звичайно": "не тепліше"
})
merged = pd.crosstab(
    climate["сезон"], climate["відхилення_2_категорії"]
).reindex(index=season_order, columns=["не тепліше", "тепліше"], fill_value=0)
chi2_merged, p_merged, dof_merged, merged_expected_array = chi2_contingency(merged)
merged_expected = pd.DataFrame(
    merged_expected_array, index=merged.index, columns=merged.columns
)
print("\nОб’єднана таблиця:")
display(merged)
print("Очікувані частоти після об’єднання:")
display(merged_expected)
print("Усі E ≥ 5:", bool((merged_expected >= 5).all().all()))
print(f"χ² = {chi2_merged:.4f}; df = {dof_merged}; p-value = {p_merged:.9f}")
print("Рішення:", "відхиляємо H₀" if p_merged < alpha else "не відхиляємо H₀")
assert (merged_expected >= 5).all().all()

Матриця клітинок з очікуваною частотою < 5:


відхилення_від_норми,холодніше,звичайно,тепліше
сезон,,,
зима,False,True,False
весна,False,True,False
літо,False,True,False
осінь,False,True,False


Кількість клітинок з E < 5: 4
  зима / звичайно: E = 2.0
  весна / звичайно: E = 2.0
  літо / звичайно: E = 2.0
  осінь / звичайно: E = 2.0

Об’єднана таблиця:


відхилення_2_категорії,не тепліше,тепліше
сезон,,
зима,12,0
весна,8,4
літо,0,12
осінь,8,4


Очікувані частоти після об’єднання:


відхилення_2_категорії,не тепліше,тепліше
сезон,,
зима,7.0,5.0
весна,7.0,5.0
літо,7.0,5.0
осінь,7.0,5.0


Усі E ≥ 5: True
χ² = 26.0571; df = 3; p-value = 0.000009278
Рішення: відхиляємо H₀


**Письмова відповідь.** У початковій таблиці **4 клітинки** стовпця «звичайно» мають очікувану частоту **2,0**, тобто порушують умову E ≥ 5. Після об’єднання «звичайно» та «холодніше» у категорію «не тепліше» таблиця стала 4 × 2, а всі очікувані частоти дорівнюють **7 або 5**.

Оновлений критерій: **χ² ≈ 26,057**, **p ≈ 0,00000928**; при α = 0,05 **відхиляємо H₀**. Отже, навіть після коректного об’єднання категорій сезон і температурна категорія статистично пов’язані в навчальному наборі. Порівняно з початковим тестом змінилися χ², ступені вільності та p-value, але **підсумкове рішення залишилося тим самим**.

## Завдання 4. Критерій узгодженості для сезонів

**H₀:** пропорції чотирьох сезонів однакові: по 25%.  
**H₁:** хоча б одна сезонна частка відрізняється від 25%.

За 4 роки маємо 48 рядків, тож теоретично очікуємо по 12 спостережень для кожного сезону.

In [5]:
season_counts = climate["сезон"].value_counts().reindex(season_order)
season_expected = np.full(4, len(climate) / 4)
result_seasons = chisquare(f_obs=season_counts.to_numpy(), f_exp=season_expected)
comparison_seasons = pd.DataFrame({
    "спостережено": season_counts.values,
    "очікувано": season_expected,
}, index=season_order)
display(comparison_seasons)
print(f"χ² = {result_seasons.statistic:.4f}; p-value = {result_seasons.pvalue:.4f}")
print("Рішення:", "відхиляємо H₀" if result_seasons.pvalue < alpha else "не відхиляємо H₀")

,спостережено,очікувано
зима,12,12.0
весна,12,12.0
літо,12,12.0
осінь,12,12.0


χ² = 0.0000; p-value = 1.0000
Рішення: не відхиляємо H₀


**Письмова відповідь.** Фактичні й очікувані частоти повністю збігаються: **12/12/12/12**. Тому χ² = **0** і p-value = **1,0** — підстав відхиляти H₀ немає. Такий результат не випадковий: генератор навмисно включив **кожен місяць усіх чотирьох років**, тобто кожен сезон представлений однаковою кількістю рядків. Це показує узгодженість із заданою схемою побудови набору, а не доказ рівномірності випадкового природного процесу.

## Завдання 5. Критерій узгодженості з власною заявленою пропорцією

**Моє припущення (H₀):** у кліматичному наборі **40%** місяців «холодніше», **20%** «звичайно» і **40%** «тепліше» від середньорічної норми.  
**H₁:** реальні частки не відповідають пропорції 40% / 20% / 40%.

**Обґрунтування до обчислення тесту.** Річне коливання має амплітуду 14 °C, тоді як інтервал «звичайно» — лише ±3 °C від бази. Отже, більшу частину року місячні температури мають опинятись по один або інший бік вузької смуги «звичайно». Оскільки косинусоїда симетрична, для «холодніше» і «тепліше» очікуємо приблизно однакові частки. 40/20/40 — проста округлена модель такого очікування, а не частки, отримані після підглядання у `value_counts()`.

In [6]:
hypothesized_proportions = pd.Series(
    [0.40, 0.20, 0.40], index=cat_order, name="заявлена_частка"
)
assert np.isclose(hypothesized_proportions.sum(), 1.0)
observed_categories = climate["відхилення_від_норми"].value_counts().reindex(cat_order, fill_value=0)
expected_categories = hypothesized_proportions * len(climate)

category_comparison = pd.DataFrame({
    "спостережено": observed_categories,
    "очікувано": expected_categories,
    "фактична_частка": (observed_categories / len(climate)).round(4),
    "заявлена_частка": hypothesized_proportions,
})
display(category_comparison)
result_own = chisquare(f_obs=observed_categories, f_exp=expected_categories)
print("Мінімальна очікувана частота:", expected_categories.min())
print(f"χ² = {result_own.statistic:.6f}; p-value = {result_own.pvalue:.6f}")
print("Рішення:", "відхиляємо H₀" if result_own.pvalue < alpha else "не відхиляємо H₀")
assert (expected_categories >= 5).all()

,спостережено,очікувано,фактична_частка,заявлена_частка
холодніше,20,19.2,0.4167,0.4
звичайно,8,9.6,0.1667,0.2
тепліше,20,19.2,0.4167,0.4


Мінімальна очікувана частота: 9.600000000000001
χ² = 0.333333; p-value = 0.846482
Рішення: не відхиляємо H₀


**Письмова відповідь.** У наборі спостерігаємо **20 «холодніше», 8 «звичайно», 20 «тепліше»**. За гіпотезою 40/20/40 очікували **19,2 / 9,6 / 19,2**. Усі очікувані частоти ≥ 5, тому стандартна умова застосовності виконана.

Статистика χ² ≈ **0,3333**, p-value ≈ **0,8465**. Оскільки p > 0,05, **не відхиляємо H₀**: спостережені частоти узгоджуються із заявленими пропорціями в межах цього тесту. Велике p-value **не доводить**, що пропорція 40/20/40 є точною або єдино правильною.

## Відповіді на контрольні питання

**1. Чому в статистиці χ² квадрат різниці та ділення на очікувану частоту?**  
Критерій обчислюється як $\chi^2=\sum\frac{(O-E)^2}{E}$, де $O$ — спостережена, а $E$ — очікувана частота. Піднесення до квадрата не дає додатним і від’ємним відхиленням взаємно знищуватися та підсилює великі розбіжності. Ділення на $E$ враховує масштаб клітинки: різниця в 5 випадків істотніша при очікуванні 5, ніж при очікуванні 500.

**2. Як обчислити очікувану частоту в таблиці спряженості?**  
За H₀ про незалежність $E_{ij}=\frac{R_i C_j}{N}$, де $R_i$ — сума рядка, $C_j$ — сума стовпця, $N$ — загальна кількість спостережень. За незалежності $P(A\cap B)=P(A)P(B)$, а отже очікувана частота перетину категорій дорівнює $N\cdot (R_i/N)\cdot(C_j/N)$. Наприклад, для «зима — холодніше» маємо $12\cdot20/48=5$.

**3. Чому велике p-value не доводить незалежність?**  
Велике p-value означає, що за умови істинності H₀ спостережений результат не є незвичним відповідно до обраного критерію. Це лише відсутність достатніх підстав **відхилити** H₀, а не доказ її істинності: даних може бути замало для виявлення реального зв’язку. Це особливо важливо для Завдань 4–5, де ми не відхилили задані пропорції.

**4. Що робити, коли очікувана частота менша за 5?**  
Стандартне χ²-наближення може стати ненадійним для розріджених таблиць. Слід перелічити проблемні клітинки й, **якщо об’єднання змістовно виправдане**, об’єднати категорії та повторити тест. У нас усі чотири очікувані частоти стовпця «звичайно» дорівнювали 2; об’єднання з «холодніше» дало таблицю 4×2 із мінімальною очікуваною частотою 5. Якщо категорії об’єднувати не можна, слід розглянути точний або симуляційний метод, чи зібрати більшу вибірку. Об’єднання не варто підбирати лише заради бажаного p-value.

## Підсумковий висновок

Для варіанта **№5 (Дніпро)** побудовано **48 синтетичних спостережень**, таблиці `pd.crosstab()` і нормованих часток. Початкова таблиця незалежності мала **4 очікувані частоти < 5**, тому категорії об’єднано та критерій χ² повторено коректніше. Після об’єднання сезон пов’язаний із тим, чи температура перевищує базову норму (p ≈ 0,00000928). Рівномірний розподіл сезонів має p = 1, а обґрунтована частка 40/20/40 для температурних категорій не суперечить даним (p ≈ 0,8465). Висновки стосуються **навчальних згенерованих даних**, а не фактичного клімату Дніпра.

### Підсумкова програмна перевірка

In [7]:
assert climate.shape[0] == 48
assert observed.to_numpy().sum() == len(climate)
assert (relative_by_season.sum(axis=1).round(12) == 1).all()
assert len(low_expected_cells) == 4
assert (merged_expected.to_numpy() >= 5).all()
assert result_seasons.pvalue == 1.0
assert np.isclose(observed_categories.sum(), 48)
assert p_initial < alpha and p_merged < alpha and result_own.pvalue > alpha
print("✓ Усі завдання та ключові перевірки виконано. Варіант №5 — Дніпро.")

✓ Усі завдання та ключові перевірки виконано. Варіант №5 — Дніпро.
